In [1]:
import csv
import nibabel as nib
import matplotlib.pyplot as plt
import random
import torch
import os
import numpy as np
from model1 import CNN_3D,NiiDataset,MultiModalTransformer,NeuralNet,TransEModel,KGMultiModalTransformer
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import nibabel as nib
import shutil
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import math
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd

In [2]:
path_existence = []
data_normal=[]
data_ad=[]
data_mci=[]
count_ad=0
count_no=0
count_mci=0
with open('NC.csv', mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)  
    for row in csv_reader:
        path = 'NC/' + row[1]
        exists = os.path.exists(path)
        path_existence.append((path, exists))
        if exists:
            count_no=count_no+1
            data_normal.append(row)
            
with open('AD.csv', mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader) 
    for row in csv_reader:
        path = 'AD/' + row[1]
        exists = os.path.exists(path)
        path_existence.append((path, exists))
        if exists:
            count_ad=count_ad+1
            data_ad.append(row)
            
with open('MCI.csv', mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader) 
    for row in csv_reader:
        path = 'MCI/' + row[1]
        exists = os.path.exists(path)
        path_existence.append((path, exists))
        if exists:
            count_mci=count_mci+1
            data_mci.append(row)
print(count_ad) #44
print(count_no) #247
print(count_mci) #106

72
358
86


In [3]:
ad_arrays=[]
replace_dict = {'female': '0', 'male': '1', 'whi': '0', 'blk': '1', '': '0','no':'0','yes':'1','ans':'2','haw':'3','ind':'4','bl':'1'}
for i in data_ad:
    j= i[14:]
    j= [replace_dict.get(item, item) for item in j]
    num_list = [float(num) for num in j]
    ad_array = np.array(num_list)
    ad_arrays.append(ad_array)
ad_array = np.vstack(ad_arrays)

In [4]:
normal_arrays=[]
for i in data_normal:
    j= i[14:]
    j= [replace_dict.get(item, item) for item in j]
    num_list = [float(num) for num in j]
    normal_array = np.array(num_list)
    normal_arrays.append(normal_array)
normal_array = np.vstack(normal_arrays)

In [5]:
mci_arrays=[]
for i in data_mci:
    j= i[14:]
    j= [replace_dict.get(item, item) for item in j]
    num_list = [float(num) for num in j]
    mci_array = np.array(num_list)
    mci_arrays.append(mci_array)
mci_array = np.vstack(mci_arrays)

In [6]:
#加权算值
def weighted_sum(tensor):
    weights = [0.2, 0.3, 0.5]
    weight_tensor = torch.tensor(weights, dtype=tensor.dtype, device=tensor.device)
    weighted_sum_result = torch.sum(tensor * weight_tensor, dim=1, keepdim=True)
    return weighted_sum_result

In [7]:
ad_tensor = torch.from_numpy(ad_array).float()
normal_tensor = torch.from_numpy(normal_array).float()
mci_tensor = torch.from_numpy(mci_array).float()
normal_labels = torch.zeros(normal_tensor.shape[0], dtype=torch.long)
mci_labels = torch.ones(mci_tensor.shape[0], dtype=torch.long)
ad_labels = torch.full((ad_tensor.shape[0],), 2, dtype=torch.long)

X = torch.cat([ad_tensor, normal_tensor, mci_tensor], dim=0)
y = torch.cat([ad_labels, normal_labels, mci_labels], dim=0)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
model = NeuralNet(embedding=9)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
for epoch in range(50):
    for inputs, labels in dataloader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
with torch.no_grad():
    ad_tensor = model(ad_tensor)
    normal_tensor = model(normal_tensor)
    mci_tensor = model(mci_tensor)
    
ad_tensor = weighted_sum(ad_tensor)
mci_tensor = weighted_sum(mci_tensor)
normal_tensor = weighted_sum(normal_tensor)

print('AD_tensor shape:', ad_tensor.shape)
print('Normal_tensor shape:', normal_tensor.shape)
print('MCI_tensor shape:', mci_tensor.shape)

AD_tensor shape: torch.Size([72, 1])
Normal_tensor shape: torch.Size([358, 1])
MCI_tensor shape: torch.Size([86, 1])


In [8]:
# 数据处理函数
def preprocess_data(data, replace_dict):
    processed_data = []
    for row in data:
        row = [replace_dict.get(item, item) for item in row]
        row = [float(item) if item.replace('.', '', 1).isdigit() else item for item in row]
        processed_data.append(row[3:18])
    return np.array(processed_data)

In [9]:
# 编码类别型变量
def encode_categorical(data, categorical_indices):
    encoded_data = data.copy()
    for idx in categorical_indices:
        le = LabelEncoder()
        encoded_data[:, idx] = le.fit_transform(encoded_data[:, idx])
    return encoded_data.astype(float)

In [10]:
ad_data = preprocess_data(data_ad, replace_dict)
normal_data = preprocess_data(data_normal, replace_dict)
mci_data = preprocess_data(data_mci, replace_dict)

categorical_indices = [2, 3, 4, 5]  # gender, education, hispanic, race, apoe
ad_EHR = encode_categorical(ad_data, categorical_indices)
normal_EHR = encode_categorical(normal_data, categorical_indices)
mci_EHR = encode_categorical(mci_data, categorical_indices)

ad_EHR = torch.from_numpy(ad_EHR).float()
normal_EHR = torch.from_numpy(normal_EHR).float()
mci_EHR = torch.from_numpy(mci_EHR).float()

linear_layer = nn.Linear(15, 8)
normal_EHR = linear_layer(normal_EHR)
ad_EHR = linear_layer(ad_EHR)
mci_EHR = linear_layer(mci_EHR)

linear_layer = nn.Linear(8, 1)
normal_EHR = linear_layer(normal_EHR)
ad_EHR = linear_layer(ad_EHR)
mci_EHR = linear_layer(mci_EHR)

print('ad.EHR--->',ad_EHR.shape)
print('normal.EHR--->',normal_EHR.shape)
print('mci.EHR--->',mci_EHR.shape)

ad.EHR---> torch.Size([72, 1])
normal.EHR---> torch.Size([358, 1])
mci.EHR---> torch.Size([86, 1])


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
# 先导入模型定义（你cell1里已import）
nii = CNN_3D(num_class=1)    # 输出维度根据你的网络结构和任务来设
nii = nii.to(device)

def get_nii_file_list(folder_path):
    import os
    return [os.path.join(folder_path, f) for f in os.listdir(folder_path)
            if f.endswith('.nii') or f.endswith('.nii.gz')]

# 加在Cell12前，假设你的CNN_3D模型叫 nii，且device设置好了
# --- AD ---
ad_nii_files = get_nii_file_list('AD')
dataset = NiiDataset(ad_nii_files)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)
all_outputs = []
for batch_idx, batch_data in enumerate(dataloader):
    batch_data = batch_data.to(device)
    output = nii(batch_data)
    all_outputs.append(output.cpu())
ad_output = torch.cat(all_outputs, dim=0)

# --- NC ---
normal_nii_files = get_nii_file_list('NC')
dataset = NiiDataset(normal_nii_files)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)
all_outputs = []
for batch_idx, batch_data in enumerate(dataloader):
    batch_data = batch_data.to(device)
    output = nii(batch_data)
    all_outputs.append(output.cpu())
normal_output = torch.cat(all_outputs, dim=0)

# --- MCI ---
mci_nii_files = get_nii_file_list('MCI')
dataset = NiiDataset(mci_nii_files)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)
all_outputs = []
for batch_idx, batch_data in enumerate(dataloader):
    batch_data = batch_data.to(device)
    output = nii(batch_data)
    all_outputs.append(output.cpu())
mci_output = torch.cat(all_outputs, dim=0)

print('ad_output shape:', ad_output.shape)
print('normal_output shape:', normal_output.shape)
print('mci_output shape:', mci_output.shape)


ad_output shape: torch.Size([72, 1])
normal_output shape: torch.Size([358, 1])
mci_output shape: torch.Size([86, 1])


In [13]:


# 获取EHR主键顺序（假设 row[1] 是nii文件名，不带扩展名）
ad_ehr_keys = [os.path.splitext(row[1])[0] for row in data_ad]
mci_ehr_keys = [os.path.splitext(row[1])[0] for row in data_mci]
normal_ehr_keys = [os.path.splitext(row[1])[0] for row in data_normal]

# NII文件主键到索引
ad_nii_files = get_nii_file_list('AD')
ad_nii_key2idx = {os.path.splitext(os.path.basename(f))[0]: i for i, f in enumerate(ad_nii_files)}
mci_nii_files = get_nii_file_list('MCI')
mci_nii_key2idx = {os.path.splitext(os.path.basename(f))[0]: i for i, f in enumerate(mci_nii_files)}
normal_nii_files = get_nii_file_list('NC')
normal_nii_key2idx = {os.path.splitext(os.path.basename(f))[0]: i for i, f in enumerate(normal_nii_files)}

# 对齐NII输出，只保留EHR有的部分（顺序与EHR一致）
ad_nii_indices = [ad_nii_key2idx[k] for k in ad_ehr_keys if k in ad_nii_key2idx]
ad_output_aligned = ad_output[ad_nii_indices]

mci_nii_indices = [mci_nii_key2idx[k] for k in mci_ehr_keys if k in mci_nii_key2idx]
mci_output_aligned = mci_output[mci_nii_indices]

normal_nii_indices = [normal_nii_key2idx[k] for k in normal_ehr_keys if k in normal_nii_key2idx]
normal_output_aligned = normal_output[normal_nii_indices]

# 检查对齐后shape
print('ad_EHR:', ad_EHR.shape, 'ad_output_aligned:', ad_output_aligned.shape)
print('mci_EHR:', mci_EHR.shape, 'mci_output_aligned:', mci_output_aligned.shape)
print('normal_EHR:', normal_EHR.shape, 'normal_output_aligned:', normal_output_aligned.shape)


ad_EHR: torch.Size([72, 1]) ad_output_aligned: torch.Size([72, 1])
mci_EHR: torch.Size([86, 1]) mci_output_aligned: torch.Size([86, 1])
normal_EHR: torch.Size([358, 1]) normal_output_aligned: torch.Size([358, 1])


In [14]:
import os

def get_nii_file_list(folder_path):
    return [os.path.join(folder_path, f) for f in os.listdir(folder_path)
            if f.endswith('.nii') or f.endswith('.nii.gz')]

batch_size = 16

# AD
ad_nii_files = get_nii_file_list('AD')
dataset = NiiDataset(ad_nii_files)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
all_outputs = []
for batch_idx, batch_data in enumerate(dataloader):
    batch_data = batch_data.to(device)
    output = nii(batch_data)
    all_outputs.append(output)
ad_output = torch.cat(all_outputs, dim=0)
print('ad nii shape--->', ad_output.shape)

# NC
normal_nii_files = get_nii_file_list('NC')
dataset = NiiDataset(normal_nii_files)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
all_outputs = []
for batch_idx, batch_data in enumerate(dataloader):
    batch_data = batch_data.to(device)
    output = nii(batch_data)
    all_outputs.append(output)
normal_output = torch.cat(all_outputs, dim=0)
print('normal nii shape--->', normal_output.shape)

# MCI
mci_nii_files = get_nii_file_list('MCI')
dataset = NiiDataset(mci_nii_files)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
all_outputs = []
for batch_idx, batch_data in enumerate(dataloader):
    batch_data = batch_data.to(device)
    output = nii(batch_data)
    all_outputs.append(output)
mci_output = torch.cat(all_outputs, dim=0)
print('mci nii shape--->', mci_output.shape)


ad nii shape---> torch.Size([72, 1])
normal nii shape---> torch.Size([358, 1])
mci nii shape---> torch.Size([86, 1])


In [15]:
import json
import os
import pandas as pd
import numpy as np
import torch

# --- 1. 读取数据  ---
ad_df = pd.read_csv('AD.csv')
mci_df = pd.read_csv('MCI.csv')
normal_df = pd.read_csv('NC.csv')

# --- 2. 加载字典和 Embedding  ---
entity2id_path = 'aibl_kg_entity2id.json' 
if not os.path.exists(entity2id_path):
    print(f"Error: {entity2id_path} not found!")
    # 模拟数据用于调试
    entity2id = {"Patient:123": 0, "Patient:456": 1} 
else:
    with open(entity2id_path, 'r', encoding='utf-8') as f:
        entity2id = json.load(f)

embed_path = 'aibl_kg_embeddings.npy'
if not os.path.exists(embed_path):
    # 模拟数据用于调试
    kg_embeddings = torch.randn(len(entity2id), 128)
else:
    distmult_matrix = np.load(embed_path)
    kg_embeddings = torch.from_numpy(distmult_matrix).float()

# ==========================================
# ★★★ 诊断代码开始 (Diagnostic Block) ★★★
# ==========================================
print("\n" + "="*30)
print("       开始诊断 NC 匹配失败原因       ")
print("="*30)

# 1. 查看 NC.csv 里的 filename 到底长什么样
print("\n[1] NC.csv 前 3 行 filename 数据:")
print(normal_df['filename'].head(3).tolist())

# 2. 查看 KG 里的 Key 到底长什么样 (打印前 5 个包含 'Patient' 的 key)
print("\n[2] KG 中的 Key 示例 (前 5 个):")
sample_keys = [k for k in list(entity2id.keys()) if "Patient" in k][:5]
print(sample_keys)

# 3. 模拟一次匹配过程，看看具体差异
sample_nc = str(normal_df['filename'].iloc[0]).strip()
constructed_key = f"Patient:{sample_nc}"
print(f"\n[3] 尝试匹配 NC 的第一个病人:")
print(f"   CSV中的原文件名: '{sample_nc}'")
print(f"   构建出的 Key   : '{constructed_key}'")

if constructed_key in entity2id:
    print(f"   ---> 结果: 匹配成功 (In KG)")
else:
    print(f"   ---> 结果: 匹配失败 (Not in KG)")
    # 尝试智能搜索：在 KG 里找找看有没有类似的
    # 比如：去掉 .nii 后缀试试？
    name_no_ext = sample_nc.replace('.nii', '').replace('.gz', '')
    guess_key = f"Patient:{name_no_ext}"
    if guess_key in entity2id:
        print(f"   [!] 发现潜在匹配: KG 中存在 '{guess_key}'，看来你需要去掉文件后缀！")
    else:
        # 看看是不是 CSV 文件名本身就是 ID 的一部分
        partial_matches = [k for k in sample_keys if name_no_ext in k]
        if partial_matches:
            print(f"   [!] KG 中类似的 Key: {partial_matches}")
        else:
            print(f"   [!] 在 KG 样本中完全找不到类似 ID。")

print("="*30 + "\n")

# ==========================================
# ★★★ 修复后的函数 (Robust Version) ★★★
# ==========================================

def get_distmult_embeddings_robust(df, entity2id, embeddings_tensor, label_name):
    embeddings_list = []
    found_count = 0
    dim = embeddings_tensor.shape[1]
    
    # 获取所有 KG 的 key 集合，用于快速查找 (如果 key 很长，这步会耗内存，但在 Jupyter 里通常没事)
    # kg_keys_set = set(entity2id.keys()) 
    
    for index, row in df.iterrows():
        fname = str(row['filename']).strip()
        
        # --- 尝试 1: 直接拼接 (针对 AD/MCI 可能有效) ---
        kg_key = f"Patient:{fname}"
        
        # --- 尝试 2: 去掉 .nii 或 .nii.gz 后缀 (针对 NC 可能有效) ---
        # 很多时候 CSV 里带后缀，但 KG ID 是纯编号
        fname_clean = fname.replace('.nii.gz', '').replace('.nii', '')
        kg_key_clean = f"Patient:{fname_clean}"
        
        idx = -1
        
        if kg_key in entity2id:
            idx = entity2id[kg_key]
        elif kg_key_clean in entity2id:
            idx = entity2id[kg_key_clean]
        
        if idx != -1:
            emb = embeddings_tensor[idx]
            embeddings_list.append(emb)
            found_count += 1
        else:
            # 没找到，填 0
            embeddings_list.append(torch.zeros(dim)) 

    print(f"Processing {label_name}...")
    print(f"  - Matched {found_count} / {len(df)} patients in KG.")
    return torch.stack(embeddings_list)

# 使用修复后的函数重新运行
ad_kg = get_distmult_embeddings_robust(ad_df, entity2id, kg_embeddings, "AD")
mci_kg = get_distmult_embeddings_robust(mci_df, entity2id, kg_embeddings, "MCI")
normal_kg = get_distmult_embeddings_robust(normal_df, entity2id, kg_embeddings, "NC") # 这里应该不再是 0 了

print(f"AD KG shape: {ad_kg.shape}")
print(f"MCI KG shape: {mci_kg.shape}")
print(f"NC KG shape: {normal_kg.shape}")


       开始诊断 NC 匹配失败原因       

[1] NC.csv 前 3 行 filename 数据:
['AIBL_1467_MR_MPRAGE_ADNI_confirmed__br_raw_20141107112943194_24_S236160_I451428.nii', 'AIBL_551_MR_MPRAGE_ADNI_confirmed__br_raw_20100514133422478_54_S84920_I173695.nii', 'AIBL_796_MR_MPRAGE_ADNI_confirmed__br_raw_20100330105154047_68_S82553_I169861.nii']

[2] KG 中的 Key 示例 (前 5 个):
['Patient:AIBL_1000_MR_MPRAGE_ADNI_confirmed__br_raw_20090107130517522_1_S61337_I132872.nii', 'Patient:AIBL_1001_MR_MPRAGE_ADNI_confirmed__br_raw_20090107131125301_1_S61338_I132873.nii', 'Patient:AIBL_100_MR_MPRAGE_ADNI_confirmed__br_raw_20090114132851679_1_S61721_I133615.nii', 'Patient:AIBL_1013_MR_MPRAGE_ADNI_confirmed__br_raw_20100518160535474_24_S85107_I173974.nii', 'Patient:AIBL_1020_MR_MPRAGE_ADNI_confirmed__br_raw_20141127100940270_106_S237660_I455196.nii']

[3] 尝试匹配 NC 的第一个病人:
   CSV中的原文件名: 'AIBL_1467_MR_MPRAGE_ADNI_confirmed__br_raw_20141107112943194_24_S236160_I451428.nii'
   构建出的 Key   : 'Patient:AIBL_1467_MR_MPRAGE_ADNI_confirmed__br_

In [16]:
# Cell 16 - Modified
kg_embed_dim = 128 # 对应你 DistMult 设置的 128

# 注意：这里把 ad_transe 换成了 ad_kg
X_ad = torch.cat([ad_EHR, ad_output_aligned.cpu(), ad_tensor, ad_kg], dim=1)
X_mci = torch.cat([mci_EHR, mci_output_aligned.cpu(), mci_tensor, mci_kg], dim=1)
X_normal = torch.cat([normal_EHR, normal_output_aligned.cpu(), normal_tensor, normal_kg], dim=1)

# 下面的代码基本不用动，只要确保分割时取对了维度
y_ad = torch.ones(len(X_ad)) * 2
y_mci = torch.ones(len(X_mci)) * 1
y_normal = torch.ones(len(X_normal)) * 0

X = torch.cat([X_ad, X_mci, X_normal], dim=0).float()
y = torch.cat([y_ad, y_mci, y_normal], dim=0).float()

# 分离特征和KG嵌入
features = X[:, :-kg_embed_dim]      
kg_embeddings_data = X[:, -kg_embed_dim:]  

# 切分数据集 (把 transe_embeddings 换成 kg_embeddings_data)
X_train, X_test, y_train, y_test, kg_train, kg_test = train_test_split(
    features.detach().numpy(), y.numpy(), kg_embeddings_data.detach().numpy(),
    test_size=0.20, stratify=y.numpy(), random_state=32
)
X_train, X_val, y_train, y_val, kg_train, kg_val = train_test_split(
    X_train, y_train, kg_train,
    test_size=0.20, stratify=y_train, random_state=30
)

# 转 Tensor
X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.LongTensor(y_train).to(device)
kg_train_tensor = torch.FloatTensor(kg_train).to(device) # DistMult 向量

X_val_tensor = torch.FloatTensor(X_val).to(device)
y_val_tensor = torch.LongTensor(y_val).to(device)
kg_val_tensor = torch.FloatTensor(kg_val).to(device)

X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.LongTensor(y_test).to(device)
kg_test_tensor = torch.FloatTensor(kg_test).to(device)

# 封装 DataLoader
train_dataset = TensorDataset(X_train_tensor, kg_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, kg_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, kg_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [17]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_probs = []
    all_labels = []
    all_preds = []
    
    for inputs, transe_embed, labels in loader:
        inputs, transe_embed, labels = inputs.to(device), transe_embed.to(device), labels.to(device)
        labels = labels.long()
        
        optimizer.zero_grad()
        outputs = model(inputs, transe_embed)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

        probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
        preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)

    # Convert lists to numpy arrays for easier manipulation
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    train_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    train_f1 = f1_score(all_labels, all_preds, average='macro')
    train_recall = recall_score(all_labels, all_preds, average='macro')
    train_precision = precision_score(all_labels, all_preds, average='macro')

    avg_loss = total_loss / len(loader)
    return avg_loss, train_auc, train_f1, train_recall, train_precision

def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs = []
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for inputs, transe_embed, labels in loader:
            inputs, transe_embed, labels = inputs.to(device), transe_embed.to(device), labels.to(device)
            labels = labels.long()
            
            outputs = model(inputs, transe_embed)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
            preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
    
    
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # 计算指标
    val_auc = roc_auc_score(all_labels, all_probs, multi_class='ovo')
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    val_recall = recall_score(all_labels, all_preds, average='macro')
    val_precision = precision_score(all_labels, all_preds, average='macro')
    
    # 计算平均损失
    avg_loss = total_loss / len(loader)
    return avg_loss, val_auc, val_f1, val_recall, val_precision


In [22]:

# Cell 18 - Modified
embed_dim = 32
transe_embed_dim = 128 # 这里其实是 DistMult dim，但名字为了兼容 model1 没改
num_epochs = 200
learning_rate = 1e-5
weight_decay = 1e-3

# --- 删除掉了原本在这里的 TransEModel 初始化和 load_state_dict ---
# 因为 Embedding 已经作为数据输入了，不需要模型来生成

# 初始化主模型
model = KGMultiModalTransformer(embed_dim=embed_dim, transe_embed_dim=transe_embed_dim).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)


train_losses = []
train_aucs = []
train_f1s = []
train_recalls = []
train_precisions = []
test_losses = []
test_aucs = []
test_f1s = []
test_recalls = []
test_precisions = []

for epoch in range(num_epochs):
    train_loss, train_auc, train_f1, train_recall, train_precision = train_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_auc, test_f1, test_recall, test_precision = validate_epoch(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    train_aucs.append(train_auc)
    train_f1s.append(train_f1)
    train_recalls.append(train_recall)
    train_precisions.append(train_precision)
    
    test_losses.append(test_loss)
    test_aucs.append(test_auc)
    test_f1s.append(test_f1)
    test_recalls.append(test_recall)
    test_precisions.append(test_precision)
    
    print(f"Epoch {epoch + 1}/{num_epochs}, "
          f"Train Loss: {train_loss:.4f}, Train AUC: {train_auc:.4f}  "   
          f"test Loss: {test_loss:.4f}, test AUC: {test_auc:.4f}")

D:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/200, Train Loss: 1.1716, Train AUC: 0.4740  test Loss: 1.1424, test AUC: 0.4149


D:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2/200, Train Loss: 1.1819, Train AUC: 0.4704  test Loss: 1.1402, test AUC: 0.4484


D:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 3/200, Train Loss: 1.1657, Train AUC: 0.4987  test Loss: 1.1383, test AUC: 0.4633
Epoch 4/200, Train Loss: 1.1581, Train AUC: 0.4988  test Loss: 1.1416, test AUC: 0.4752
Epoch 5/200, Train Loss: 1.1176, Train AUC: 0.5538  test Loss: 1.1389, test AUC: 0.4881
Epoch 6/200, Train Loss: 1.1566, Train AUC: 0.5153  test Loss: 1.1439, test AUC: 0.4955
Epoch 7/200, Train Loss: 1.1458, Train AUC: 0.5236  test Loss: 1.1361, test AUC: 0.5085
Epoch 8/200, Train Loss: 1.1204, Train AUC: 0.5759  test Loss: 1.1244, test AUC: 0.5208
Epoch 9/200, Train Loss: 1.1210, Train AUC: 0.5541  test Loss: 1.1254, test AUC: 0.5348
Epoch 10/200, Train Loss: 1.1026, Train AUC: 0.5754  test Loss: 1.1235, test AUC: 0.5493
Epoch 11/200, Train Loss: 1.1138, Train AUC: 0.5894  test Loss: 1.1123, test AUC: 0.5608
Epoch 12/200, Train Loss: 1.0931, Train AUC: 0.6164  test Loss: 1.1047, test AUC: 0.5691
Epoch 13/200, Train Loss: 1.1146, Train AUC: 0.5817  test Loss: 1.0950, test AUC: 0.5852
Epoch 14/200, Train Loss: 1.

In [23]:
model.eval()
all_probs = []
all_labels = []
with torch.no_grad():
    for inputs, transe_embed, labels in val_loader:
        inputs, transe_embed, labels = inputs.to(device), transe_embed.to(device), labels.to(device)
        outputs = model(inputs, transe_embed)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy())

preds = np.argmax(all_probs, axis=1)
accuracy = accuracy_score(all_labels, preds)
precision = precision_score(all_labels, preds, average='macro')
recall = recall_score(all_labels, preds, average='macro')
f1 = f1_score(all_labels, preds, average='macro')
auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')

print("\n=== Final Validation Metrics ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"AUC-ROC:   {auc:.4f}")


=== Final Validation Metrics ===
Accuracy:  0.9036
Recall:    0.9360
F1 Score:  0.8973
Precision: 0.8769
AUC-ROC:   0.9681


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(len(train_aucs)), train_aucs, label="Train AUC", color="blue")
plt.plot(range(len(test_aucs)), test_aucs, label="Test AUC", color="red")
plt.title("AIBL AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(len(train_losses)), train_losses, label="Train Loss", color="blue")
plt.plot(range(len(test_losses)), test_losses, label="Test Loss", color="red")
plt.title("AIBL Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()